General pipeline: preprocessing => segmentation => feature extraction => classification (https://peerj.com/articles/cs-620/#fig-5)

Quality improvement: Contrast stretching, Grayscale stretching, Log transformation, Gamma correction,
Image negative, Histogram equalization methods, Adaptive local contrast stretching

Filtering (Gaussian, Poisson, and Quantum noise are different types of
noise artifacts - which one do we have?? because 'If we try to minimize one
class of noise, it may disrupt the other'): Average filter,
Bilateral filter, Laplacian filter, Homomorphic filter, and Butterworth filter, Median
Gaussian filter, and Weiner filter, Median (reduces boundaries), Gaussian (reduces picture information)

Segmentation: segmentation of panoramic X-rays using wavelet transformation shows
better results than adaptive and iterative thresholding, template matching technique, Otsu’s threshold combined with morphological dilation, (gap valley extraction, modified canny
edge detector, guided iterative contour tracing, and template matching), contour-based segmentation,  horizontal integral projection, computing moments and statistical characteristics, edge segmentation methods: Canny
and Sobel, Quantum Particle Swarm Optimization (QPSO)  employed for
multilevel thresholding,  Gaussian kernel-based conditional spatial
fuzzy c-means (GK-csFCM) clustering algorithm.

Classic ML: Feature extraction (Projected principal edge distribution
(PPED) + Geometric properties +
Region descriptors) + SVM, Segmentation of mandibular teeth carried
out by applying Random forest regression-
voting constrained local model (RFRV-
CLM) in two steps: The 1st step gives an
estimate of individual teeth and mandible
regions used to initialize search for the
tooth. In the second step, the investigation
is carried out separately for each tooth.


In [1]:
import numpy as np
import cv2
import json
import os
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import kagglehub

DOWNLOAD_DATASET = False

In [2]:
if DOWNLOAD_DATASET:
    path = kagglehub.dataset_download("humansintheloop/teeth-segmentation-on-dental-x-ray-images")
    sources = {k: Path(path) / f'Teeth Segmentation {k}' for k in ['JSON', 'PNG']}
else:
    sources = {k: Path(f'./Teeth Segmentation {k}') for k in ['JSON', 'PNG']}

In [3]:
meta = {}
for p in sources.values():
    meta.update(json.loads((p / 'meta.json').read_text()))

images = [
    Image.open(img_path)
    for p in sources.values()
    for img_path in (p / 'd2' / 'img').glob('*')
]

images_np = [np.array(image) for image in images]

In [4]:
for image in images_np:
    print(image.shape)

(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2038)
(1024, 2041)
(1024, 2041)
(1024, 2038)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2043)
(1024, 1852)
(1024, 2041)
(1024, 2041)
(1024, 1850)
(1024, 2041)
(1024, 2043)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 1852)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2038)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 2041)
(1024, 1852)
(1024, 2038)
(1024, 2041)
(1024, 2041)

Suggestion:
1. preprocessing:
- CLAHE
- every noise reduction that do not blur edges (unsharp masking / bilateral filtering)
- morphological operation (erosion / dilatation etc. - check before or after first segmentation)
- validate visually + check strength of found edges (Sobel/Canny)
- Gamma correction
-  Log transformation
- Median filter, Gaussian filter, Wiener filter (check what type of noise there is primarly [SaltNPepper / Gaussian / Photon(Quantum) / ... ]
2. segmentation:
- watershed with markers on each tooth
- active contours (snake etc)
- teeth touching / overlapping = concavity analysis + find ways to count teeth properly
- Validate segmentation (compute Dice coefficient vs ground truth), IoU, visually how they look compared to manually prepared, check number of teeth
- remove very small/large segments
- Laplacian filter for edges
- Otsu's threshold + morphological dilation
- Contour-based segmentation
- Wavelet transformation
3. Feature extraction:
- hu moments, aspect ratio, solidity, circularity/compactness, eccentricity of fitted ellipse
- Position/context features: centroid position, orientation angle, number of neighbors and distances to them, relative position in dental arch
- Texture features (often overlooked but useful): Local Binary Patterns (LBP) on tooth region, Gray-level statistic
- plot how clusterization works (of types / sides / jaws)
- Projected Principal Edge Distribution + Shape Descriptors + Region descriptors (texture etc.) => SVM
4. Classification:
- two-step: tooth type classification (SVM / RF with extracted features)
- second: get specific tooth class (have type, have orientation [bottom/top jaw, left/right part of mouth])
- spatial graph of teeth, compare to template
- cross - validation
- we can even try to extract as many features to try to just generate one class (1..32) for every tooth (easier if we want to see some results)
- graph data: centroid position, tooth type, jaw (top/bottom), neighbors (teeth within certain distance)
- prepare template from all images (average / median positions by teeth type+class, distances, etc.)
- Random Forest estimates approximate regions for each tooth (search regions) => CLM (Constrained Local Model) searches for a tooth in every region
5. Missing teeth:
- check distances between neighbours
- compare to template to see where the tooth should be if dist > expected, mark as missing
-

<h1><b>STEP1:</b> Normalize images, resize or truncate to one size </h1>

<h1><b>STEP2:</b> Find out what type of noise is present on the images </h1>

<h1><b>STEP3:</b> Image denoising </h1>

<h1><b>STEP4:</b> Check segmentation after denoising </h1>

<h1><b>STEP5:</b> Add after segmentation processing <i>(optional)</i> </h1>

<h1><b>STEP6:</b> Extract features from segmented images </h1>

<h1><b>STEP7:</b> Model for tooth type classification </h1>

<h1><b>STEP8:</b> Prepare tooth data for template </h1>

<h1><b>STEP9:</b> Create template map </h1>

<h1><b>STEP10:</b> Generate summary of teeth status (whole pipeline + model for graph comparaison with template) </h1>